# RG3 gas: transient phase rather than fluctuation magnitude
Six predeclared features distinguish oscillation shape, age of last biggest change, and lagged thermal/cushion coupling. Not the previous count/growth features.
Current measured shot may be used; no future shot, no target label in feature extraction. Windows follow existing product/gap history boundaries.
RF+ExtraTrees mean/max; thresholds from inner held-out normals. 1/5/10% rates are diagnostic candidates; selecting outer-best is exploratory only.
Normals may share rolling windows with gas cases; causal history availability and actual production Run boundaries still require verification.

In [ ]:
from pathlib import Path
import json
import numpy as np,pandas as pd
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score,confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
old=json.loads((root/'notebooks/10_original_defect_report_revision.ipynb').read_text(encoding='utf-8'))
exec(''.join(next(c for c in old['cells'] if c['cell_type']=='code')['source']).split('for feature in features:')[0])
allshots=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
d=allshots.loc[allshots['product'].eq('RG3')&(allshots.PassOrFail.eq(0)|allshots.Reason.str.contains('가스',na=False))].sort_values('TimeStamp').reset_index(drop=True)
assert len(d)==583 and d.PassOrFail.sum()==17
y=d.PassOrFail.to_numpy(int);groups=d.comparison_block
blocks=groups.drop_duplicates().tolist()
sensors=['Barrel_Temperature_2','Hopper_Temperature','Cushion_Position']
floors=np.array([.1,.1,.01])
segments={k:g.sort_values('TimeStamp').reset_index(drop=True) for k,g in observed.loc[observed['product'].eq('RG3')].groupby('history_segment')}
names=['barrel2_roughness','hopper_roughness','cushion_roughness','hopper_peak_age','cushion_peak_age','thermal_cushion_lag_correlation']
F=np.full((len(d),6),np.nan);coverage=[]
for n,r in d.iterrows():
    g=segments[r.history_segment]
    i=int(np.searchsorted(g.TimeStamp.values,np.datetime64(r.TimeStamp),side='left'))
    assert i<len(g) and g.TimeStamp.iloc[i]==r.TimeStamp
    # Includes current measured sensors, excludes ALL future observations.
    window=g.iloc[max(0,i-10):i+1][sensors].to_numpy(float)
    valid=len(window)>=6 and np.isfinite(window).all()
    coverage.append(dict(TimeStamp=r.TimeStamp,actual=int(y[n]),window_observations=len(window),complete=bool(valid)))
    if not valid:continue
    inc=np.diff(window,axis=0)
    rough=np.std(inc,axis=0)/np.maximum(np.std(window,axis=0),floors)
    def age(col):
        a=np.abs(inc[:,col])
        # Choose most recent tied peak; recording ties are frequent.
        j=len(a)-1-int(np.argmax(a[::-1]))
        return (len(a)-1-j)/max(len(a)-1,1)
    th=inc[:-1,1];cu=inc[1:,2]
    if np.std(th)>floors[1]*.1 and np.std(cu)>floors[2]*.1:
        corr=float(np.corrcoef(th,cu)[0,1])
    else:corr=0.
    F[n]=[rough[0],rough[1],rough[2],age(1),age(2),corr]
pd.DataFrame(F,columns=names).assign(TimeStamp=d.TimeStamp).to_csv(tables/'rg3_transient_phase_features.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(coverage).to_csv(tables/'rg3_transient_phase_coverage.csv',index=False,encoding='utf-8-sig')
cache={}
def predict(ref,q):
    key=(tuple(ref),tuple(q))
    if key in cache:return cache[key]
    assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
    if len(np.unique(y[ref]))<2:return np.zeros((len(q),2))
    scores=[]
    for cls in [RandomForestClassifier,ExtraTreesClassifier]:
        model=make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),cls(n_estimators=100,max_depth=5,min_samples_leaf=3,class_weight='balanced',random_state=42,n_jobs=2))
        scores.append(model.fit(F[ref],y[ref]).predict_proba(F[q])[:,1])
    cache[key]=np.column_stack([np.mean(scores,axis=0),np.max(scores,axis=0)])
    return cache[key]
records=[];audit=[]
for scheme in ['leave_one_block_out','past_blocks_only']:
    for j,b in enumerate(blocks):
        q=np.flatnonzero(groups.eq(b));ref=np.flatnonzero(~groups.eq(b)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
        if len(np.unique(y[ref]))<2:continue
        outer=predict(ref,q);inner=np.full((len(ref),2),np.nan)
        for ib in groups.iloc[ref].unique():
            mask=groups.iloc[ref].eq(ib).to_numpy()
            if (~mask).any():inner[mask]=predict(ref[~mask],ref[mask])
        for k,name in enumerate(['phase_RF_ET_mean','phase_RF_ET_max']):
            norm=inner[(y[ref]==0)&np.isfinite(inner[:,k]),k]
            for rate in [.01,.05,.10]:
                cut=float(np.quantile(norm,1-rate,method='higher')) if len(norm) else 1.
                for l,i in enumerate(q):records.append(dict(scheme=scheme,model=name,budget=rate,TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(outer[l,k]),threshold=cut,prediction=int(outer[l,k]>cut)))
        audit.append(dict(scheme=scheme,block=b,train=len(ref),test=len(q),train_gas=int(y[ref].sum()),test_gas=int(y[q].sum())))
        print('Finished transient phase',scheme,b,flush=True)
pp=pd.DataFrame(records)
base=pd.read_csv(tables/'rg3_nested_ensemble_predictions.csv',parse_dates=['TimeStamp'])
base=base.loc[base.model.eq('max_union'),['scheme','TimeStamp','predicted_train_fpr1']]
rows=[]
for (scheme,name,rate),g in pp.groupby(['scheme','model','budget'],sort=False):
    z=g.merge(base,on=['scheme','TimeStamp'],validate='one_to_one')
    for combine in ['new_only','OR_RF','AND_RF']:
        pr=z.prediction.astype(bool)
        if combine=='OR_RF':pr=pr|z.predicted_train_fpr1
        if combine=='AND_RF':pr=pr&z.predicted_train_fpr1
        tn,fp,fn,tp=confusion_matrix(z.actual,pr,labels=[0,1]).ravel()
        rows.append(dict(scheme=scheme,model=name,budget=rate,combine=combine,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.,new_model_AP=average_precision_score(z.actual,z.score)))
metrics=pd.DataFrame(rows)
pp.to_csv(tables/'rg3_transient_phase_predictions.csv',index=False,encoding='utf-8-sig')
metrics.to_csv(tables/'rg3_transient_phase_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(audit).to_csv(tables/'rg3_transient_phase_fold_audit.csv',index=False,encoding='utf-8-sig')
print(metrics.sort_values(['scheme','F1','FP'],ascending=[True,False,True]).groupby('scheme').head(8).to_string(index=False))


## Executed results
Whole normal99 mean OR RF: TP4 FP9 F1=0.267. Relaxed max normal90 OR RF: TP7 FP74 F1=0.143. No balanced improvement; not adopted.
Only causal original sensor histories through current shot; six phase features, not a defect cause inference. See transient_phase tables.